- Curso: PPG em Genética e Biologia Molecular (PPGBM-UFPA)
- Disciplina: IA na Descoberta Genética
- Tema: Deep Learning para Genômica
- Docente: Helder Matos e Sintia Almeida

In [20]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import imblearn

# 0. Preliminares

## 0.1. Knowledge Discovery in Databases (KDD)

KDD (*Knowledge Discovery in Databases*) é o processo de descobrir padrões em dados. Padrões válidos, novos, úteis e que fazem sentido para quem olha o resultado (FAYYAD et al., 1996).

É interativo: o analista intervém em cada etapa. E iterativo: o processo volta e refina etapas anteriores conforme o entendimento avança.

Cinco fases:

1. **Seleção**: escolher quais dados e variáveis entram na análise.
2. **Pré-processamento**: limpar ruído, tratar dados ausentes.
3. **Transformação**: formatar e armazenar os dados do jeito que a ferramenta de mineração espera.
4. **Mineração de dados**: buscar os padrões propriamente ditos.
5. **Interpretação e avaliação**: visualizar os padrões encontrados e checar se eles realmente aprenderam algo útil.

<div style="text-align: center;">
    <img src="https://cdn.jsdelivr.net/gh/hellsdeur/multiomic/images/kdd-process.png" alt="Processo KDD" width="100%">
</div>

# 1. Seleção

## 1.1. Carregamento dos dados

In [4]:
df_atac = pd.read_csv("data/tables/ATAC_peak_counts.csv")
df_atac

,sample_id,GENA_promoter,GENB_promoter,GENC_promoter,GEND_promoter,HK1_promoter,HK2_promoter
0,S01,617,910,681,255,745,847
1,S02,681,710,711,244,714,805
2,S03,641,906,762,335,737,761
3,S04,684,833,665,293,836,689
4,S05,676,779,805,289,819,709
5,S06,764,875,596,384,728,734
6,S07,638,137,743,496,817,855
7,S08,670,171,726,589,688,684
8,S09,676,139,635,535,815,728
9,S10,645,171,739,538,915,784


In [5]:
df_genotypes = pd.read_csv("data/tables/genotypes.csv")
df_genotypes

,sample_id,rsToy1_GENA,rsToy2_GENB,rsToy3_GENC,rsToy4_REG
0,S01,0,0,2,1
1,S02,1,0,0,0
2,S03,0,2,1,0
3,S04,0,0,0,0
4,S05,0,2,0,2
5,S06,2,0,1,2
6,S07,1,0,0,1
7,S08,1,1,0,0
8,S09,1,0,0,1
9,S10,0,1,1,2


In [6]:
df_metadata = pd.read_csv("data/tables/metadata.csv")
df_metadata

,sample_id,group,phenotype,sex,batch,age_years,notes
0,S01,Control,0,F,B1,34,baseline phenotype
1,S02,Control,0,M,B1,41,baseline phenotype
2,S03,Control,0,F,B1,29,baseline phenotype
3,S04,Control,0,M,B2,50,baseline phenotype
4,S05,Control,0,F,B2,38,baseline phenotype
5,S06,Control,0,M,B2,45,baseline phenotype
6,S07,Deficient,1,F,B1,36,reduced product of three-gene protein complex
7,S08,Deficient,1,M,B1,43,reduced product of three-gene protein complex
8,S09,Deficient,1,F,B1,31,reduced product of three-gene protein complex
9,S10,Deficient,1,M,B2,48,reduced product of three-gene protein complex


In [7]:
df_multiomics = pd.read_csv("data/tables/multiomics_ML_ready.csv")
df_multiomics

,sample_id,phenotype,group,batch,RNA_GENA,PROT_GENA,RNA_GENB,PROT_GENB,RNA_GENC,PROT_GENC,...,ATAC_GENA,ATAC_GENB,ATAC_GENC,ATAC_GEND,ATAC_HK1,ATAC_HK2,GT_rsToy1_GENA,GT_rsToy2_GENB,GT_rsToy3_GENC,GT_rsToy4_REG
0,S01,0,Control,B1,996,1134.33,1179,1430.87,755,1053.74,...,617,910,681,255,745,847,0,0,2,1
1,S02,0,Control,B1,967,1189.38,1160,1232.65,854,1095.08,...,681,710,711,244,714,805,1,0,0,0
2,S03,0,Control,B1,991,1289.34,990,1334.15,672,1146.85,...,641,906,762,335,737,761,0,2,1,0
3,S04,0,Control,B2,917,1290.23,1267,1263.45,802,1095.92,...,684,833,665,293,836,689,0,0,0,0
4,S05,0,Control,B2,946,1278.27,996,1262.00,781,1061.68,...,676,779,805,289,819,709,0,2,0,2
5,S06,0,Control,B2,999,1132.29,1059,1392.68,878,1110.27,...,764,875,596,384,728,734,2,0,1,2
6,S07,1,Deficient,B1,900,1445.01,304,364.62,837,1159.84,...,638,137,743,496,817,855,1,0,0,1
7,S08,1,Deficient,B1,944,1404.72,228,360.33,801,1242.83,...,670,171,726,589,688,684,1,1,0,0
8,S09,1,Deficient,B1,802,1153.27,236,372.82,826,1274.07,...,676,139,635,535,815,728,1,0,0,1
9,S10,1,Deficient,B2,1041,1204.00,255,373.63,826,1094.24,...,645,171,739,538,915,784,0,1,1,2


In [8]:
df_proteomics = pd.read_csv("data/tables/proteomics_intensity.csv")
df_proteomics

,sample_id,GENA,GENB,GENC,GEND,HK1,HK2
0,S01,1134.33,1430.87,1053.74,481.60,1029.16,931.07
1,S02,1189.38,1232.65,1095.08,489.61,1039.74,960.05
2,S03,1289.34,1334.15,1146.85,487.02,1017.06,1077.44
3,S04,1290.23,1263.45,1095.92,561.81,973.28,989.07
4,S05,1278.27,1262.00,1061.68,529.04,998.58,1054.18
5,S06,1132.29,1392.68,1110.27,515.51,1040.04,1048.68
6,S07,1445.01,364.62,1159.84,1186.16,1056.43,982.24
7,S08,1404.72,360.33,1242.83,1001.16,968.70,985.71
8,S09,1153.27,372.82,1274.07,1162.20,1033.40,1023.61
9,S10,1204.00,373.63,1094.24,1051.27,1029.77,1130.74


In [9]:
df_rna = pd.read_csv("data/tables/RNA_counts.csv")
df_rna

,sample_id,GENA,GENB,GENC,GEND,HK1,HK2
0,S01,996,1179,755,230,1086,1028
1,S02,967,1160,854,248,923,930
2,S03,991,990,672,295,1027,1144
3,S04,917,1267,802,275,1054,1089
4,S05,946,996,781,236,1002,807
5,S06,999,1059,878,250,993,957
6,S07,900,304,837,636,989,1154
7,S08,944,228,801,731,1076,905
8,S09,802,236,826,696,1149,1047
9,S10,1041,255,826,681,841,900


## 1.2. Análise exploratória dos dados

In [11]:
df_multiomics.columns

Index(['sample_id', 'phenotype', 'group', 'batch', 'RNA_GENA', 'PROT_GENA',
       'RNA_GENB', 'PROT_GENB', 'RNA_GENC', 'PROT_GENC', 'RNA_GEND',
       'PROT_GEND', 'RNA_HK1', 'PROT_HK1', 'RNA_HK2', 'PROT_HK2', 'ATAC_GENA',
       'ATAC_GENB', 'ATAC_GENC', 'ATAC_GEND', 'ATAC_HK1', 'ATAC_HK2',
       'GT_rsToy1_GENA', 'GT_rsToy2_GENB', 'GT_rsToy3_GENC', 'GT_rsToy4_REG'],
      dtype='str')

In [12]:
df_multiomics

,sample_id,phenotype,group,batch,RNA_GENA,PROT_GENA,RNA_GENB,PROT_GENB,RNA_GENC,PROT_GENC,...,ATAC_GENA,ATAC_GENB,ATAC_GENC,ATAC_GEND,ATAC_HK1,ATAC_HK2,GT_rsToy1_GENA,GT_rsToy2_GENB,GT_rsToy3_GENC,GT_rsToy4_REG
0,S01,0,Control,B1,996,1134.33,1179,1430.87,755,1053.74,...,617,910,681,255,745,847,0,0,2,1
1,S02,0,Control,B1,967,1189.38,1160,1232.65,854,1095.08,...,681,710,711,244,714,805,1,0,0,0
2,S03,0,Control,B1,991,1289.34,990,1334.15,672,1146.85,...,641,906,762,335,737,761,0,2,1,0
3,S04,0,Control,B2,917,1290.23,1267,1263.45,802,1095.92,...,684,833,665,293,836,689,0,0,0,0
4,S05,0,Control,B2,946,1278.27,996,1262.00,781,1061.68,...,676,779,805,289,819,709,0,2,0,2
5,S06,0,Control,B2,999,1132.29,1059,1392.68,878,1110.27,...,764,875,596,384,728,734,2,0,1,2
6,S07,1,Deficient,B1,900,1445.01,304,364.62,837,1159.84,...,638,137,743,496,817,855,1,0,0,1
7,S08,1,Deficient,B1,944,1404.72,228,360.33,801,1242.83,...,670,171,726,589,688,684,1,1,0,0
8,S09,1,Deficient,B1,802,1153.27,236,372.82,826,1274.07,...,676,139,635,535,815,728,1,0,0,1
9,S10,1,Deficient,B2,1041,1204.00,255,373.63,826,1094.24,...,645,171,739,538,915,784,0,1,1,2


In [13]:
df_multiomics.describe()

,phenotype,RNA_GENA,PROT_GENA,RNA_GENB,PROT_GENB,RNA_GENC,PROT_GENC,RNA_GEND,PROT_GEND,RNA_HK1,...,ATAC_GENA,ATAC_GENB,ATAC_GENC,ATAC_GEND,ATAC_HK1,ATAC_HK2,GT_rsToy1_GENA,GT_rsToy2_GENB,GT_rsToy3_GENC,GT_rsToy4_REG
count,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.00000,...,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000
mean,0.500000,937.916667,1232.450833,687.666667,839.667500,796.833333,1136.161667,465.250000,807.363333,1029.75000,...,661.416667,494.500000,697.166667,414.666667,778.083333,760.583333,0.583333,0.583333,0.583333,1.000000
std,0.522233,69.331166,109.343733,446.430964,503.929322,55.106233,67.106337,221.383717,315.015892,88.13123,...,40.971073,360.243478,60.671745,127.135814,72.817903,59.310368,0.668558,0.792961,0.792961,0.852803
min,0.000000,802.000000,1126.630000,228.000000,343.260000,672.000000,1053.740000,230.000000,481.600000,841.00000,...,610.000000,137.000000,596.000000,244.000000,679.000000,684.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,912.750000,1140.037500,272.250000,363.547500,774.500000,1094.870000,249.500000,509.035000,992.00000,...,637.250000,153.250000,656.750000,292.000000,724.500000,716.500000,0.000000,0.000000,0.000000,0.000000
50%,0.500000,945.000000,1196.690000,647.000000,803.140000,801.500000,1128.360000,455.000000,781.485000,1040.50000,...,657.500000,440.500000,696.000000,436.500000,780.000000,747.500000,0.500000,0.000000,0.000000,1.000000
75%,1.000000,992.250000,1289.562500,1084.250000,1281.125000,828.750000,1154.687500,683.250000,1079.002500,1078.50000,...,677.250000,843.500000,740.000000,530.500000,823.250000,806.750000,1.000000,1.000000,1.000000,2.000000
max,1.000000,1041.000000,1445.010000,1267.000000,1430.870000,878.000000,1274.070000,731.000000,1186.160000,1149.00000,...,764.000000,910.000000,805.000000,589.000000,915.000000,855.000000,2.000000,2.000000,2.000000,2.000000


# 2. Pré-processamento

# 3. Transformação

In [17]:
non_feature_cols = ["sample_id", "phenotype", "group", "batch"]

X = df_multiomics.drop(columns=non_feature_cols)
y = df_multiomics["phenotype"]

print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")

Samples: 12, Features: 22


# 4. Mineração de dados

In [25]:
loo = LeaveOneOut()
y_pred = np.zeros_like(y)

for train_index, test_index in loo.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)

    y_pred[test_index] = model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y, y_pred)}\n")
print(f"Confusion Matrix:\n{confusion_matrix(y, y_pred)}\n")
print(f"Classification Report: {classification_report(y, y_pred)}")

Accuracy: 1.0

Confusion Matrix:
[[6 0]
 [0 6]]

Classification Report:               precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       1.00      1.00      1.00         6

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12



# 5. Interpretação e avaliação

# 6. Retorno à seleção

# 7. Simulador

# 8. Conclusão

# Referências